In [3]:
import matplotlib.pyplot as plt 
import numpy as np
import os 
import sys 

parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, parent_dir)

from utils.ppnet_utils import initialize_network, custom_se
from utils.gen_utils import load_config
from utils.load_data_utils import load_sampled_input_data

In [4]:
config = load_config('config.yaml')

# initialise the network and use the same data to solve wls and GAT for the SE 
net = initialize_network(net_name=config['data']['net_name'],
                         load_std=config['data']['load_std'])

num_samples = config['data']['num_samples']
num_buses = len(net.bus)

# use same data for all: wls, gat, tapsegnn
sampled_input_data = load_sampled_input_data(sc_type=config['data']['scenario_type'], 
                                                   net=net, 
                                                   num_samples=config['data']['num_samples'], 
                                                   noise=config['data']['noise'])


Network: net_4bus is selected 

Net net_4bus has 4 nodes and 4 edges. 

Selecting all the MV/LV transformers in the network 

>>> Using relative noise of 1.0% on voltage and 5.0% on power measurements.
Scaling inputs...
Number of V, P measurements 3 out of 8

Number of P_to, Q_to, P_from, Q_from measurements 1 out of 4

Sparsity of PV measurements at buses = 0.5%
Sparsity of P+ measurements at branches = 0.5


In [5]:
import torch
from utils.model_utils import initialize_model, get_eval_results
from src.dataset.custom_dataset import NodeEdgeTapDatasetV2
from utils.gen_utils import dataset_splitter
from training.trainer import trainer

device='cpu'

dataset = NodeEdgeTapDatasetV2(model_name=config['model']['name'], sampled_input_data=sampled_input_data)

all_loaders, plot_loader = dataset_splitter(dataset,
                                            batch_size=config['loader']['batch_size'],
                                            split_list=config['loader']['split_list'])

# use TapSEGNN 
model = initialize_model(model_name="NEGATRegressor",
                        dataset=dataset,
                        node_out_features=config['model']['node_out_features'],
                        list_node_hidden_features=config['model']['list_node_hidden_features'],
                        k_hop_node=config['model']['k_hop_node'],
                        edge_out_features=config['model']['edge_out_features'], 
                        list_edge_hidden_features=config['model']['list_edge_hidden_features'],
                        k_hop_edge=config['model']['k_hop_edge'],
                        trafo_hop=config['model']['trafo_hop'],
                        edge_index_list=sampled_input_data['edge_index'],
                        gat_out_features=config['model']['gat_out_features'],
                        gat_head=config['model']['gat_head'],
                        bias=config['model']['bias'], 
                        normalize=config['model']['normalize'], 
                        device=device,
                        ).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total number of parameters of model {model}: {total_params}')

optimizer = torch.optim.Adam(model.parameters(),
                            lr=config['training']['lr'], 
                            weight_decay=config['training']['weight_decay'])
        
schedular = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer, 
                                                    mode='min',
                                                    factor=0.1, 
                                                    patience=1,
                                                    min_lr=config['training']['schedular_min_lr'])

all_losses = trainer(model=model, 
                    train_loader=all_loaders[0], 
                    val_loader=all_loaders[1], 
                    test_loader=all_loaders[2], 
                    optimizer=optimizer,
                    schedular=schedular,
                    num_epoch=config['training']['num_epochs'],
                    early_stopping=config['training']['early_stopping'],
                    val_patience=config['training']['val_patience'], 
                    tap_weight=config['training']['loss_tap_weight'], 
                    device=device)

results_tapse, pred_se_va_tapse, label_se_va_tapse  = get_eval_results(test_loader=all_loaders[2],
                                tap_weight=config['training']['loss_tap_weight'], 
                            trained_model=model, 
                            scaler=sampled_input_data['scaler_y_label'], 
                            output_pred_va=True)

num_graphs = int(pred_se_va_tapse.shape[0]/num_buses)



Dataset for MultiTapSEGNN selected!


 Directed power flows accounted in dataset...


 get_edge_index_lu handling dictionary of tensors...

Total number of parameters of model NEGATRegressor(
  (node_layers): ModuleList(
    (0): TAGConv(2, 64, K=3)
    (1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  )
  (fc_node): Linear(in_features=64, out_features=32, bias=True)
  (edge_layers): ModuleList(
    (0): ModuleList(
      (0-1): 2 x TAGConv(6, 128, K=1)
    )
  )
  (edge_biases): ParameterList(  (0): Parameter containing: [torch.float32 of size 128])
  (fc_edge): Linear(in_features=128, out_features=64, bias=True)
  (gatconv): GATConv(32, 32, heads=1)
  (mlp_gat): Sequential(
    (0): Linear(in_features=32, out_features=2, bias=True)
  )
): 17506
At epoch: 0, 	 training loss: 9.443e+00,                     	 validation loss: 3.599e+00 	 lr: 1.00e-02 	 grad_norm: 3.915e+01
At epoch: 50, 	 training loss: 2.413e-01,                     	 validation loss: 2.952e-01 	 lr: 1.00e-04

In [6]:
# remove offset 
pred_se_va = pred_se_va_tapse - 10.0 
label_se_va = label_se_va_tapse - 10.0 

pred_se_va_rs = torch.reshape(pred_se_va, (num_graphs,num_buses,-1))
label_se_va_rs = torch.reshape(label_se_va, (num_graphs,num_buses,-1))

# Calculate errors for voltage (index 0) and angle (index 1) 
error_v = pred_se_va_rs[:, :, 0] - label_se_va_rs[:, :, 0]  # Shape: (batch_size, num_buses)
error_a = pred_se_va_rs[:, :, 1] - label_se_va_rs[:, :, 1]  # Shape: (batch_size, num_buses)

error_v = error_v ** 2
error_a = error_a ** 2 

mean_error_v = error_v.mean(dim=0)
tapsegnn_rmse_v = torch.sqrt(mean_error_v)

mean_error_a = error_a.mean(dim=0)
tapsegnn_rmse_a = torch.sqrt(mean_error_a)

In [ ]:
# torch.save(model.state_dict(), parent_dir + "/results/best_SEmodel.pth")
